# 🚀 MICKEY JAGGER - AI Avatar Backend

**Step 1:** Click ▶️ Run on cell 1
**Step 2:** Click ▶️ Run on cell 2
**Step 3:** Click ▶️ Run on cell 3 (gets URL!)

⚠️ Keep this tab OPEN!

In [ ]:
# 📦 STEP 1: Install Requirements
!pip install fastapi uvicorn python-multipart websockets pillow numpy pyngrok insightface torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q
print("✅ INSTALLED!")

In [ ]:
# 📥 STEP 2: Clone LivePortrait & Download Models
!git clone --depth 1 https://github.com/KwaiVGI/LivePortrait.git /content/LivePortrait

# Create folders
!mkdir -p /content/LivePortrait/pretrained_weights/liveportrait/base_models

# Download from HuggingFace
!pip install huggingface_hub -q
from huggingface_hub import hf_hub_download

files = [
    ("appearance_feature_extractor.pth", "appearance_feature_extractor.pth"),
    ("motion_extractor.pth", "motion_extractor.pth"),
    ("warping_module.pth", "warping_module.pth"),
    ("spade_generator.pth", "spade_generator.pth")
]

save_dir = "/content/LivePortrait/pretrained_weights/liveportrait/base_models"
for fname, _ in files:
    print(f"Downloading {fname}...")
    try:
        hf_hub_download(
            repo_id="KwaiVGI/LivePortrait",
            filename=f"pretrained_weights/liveportrait/base_models/{fname}",
            local_dir=save_dir,
            local_dir_use_symlinks=False
        )
        print(f"✅ {fname}")
    except Exception as e:
        print(f"⚠️ {fname}: {e}")

!ls -la /content/LivePortrait/pretrained_weights/liveportrait/base_models/
print("\n✅ MODELS READY!")

In [ ]:
# 🟢 STEP 3: Start Server

# 1️⃣ Get FREE ngrok token: https://dashboard.ngrok.com
NGROK_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"  # <-- PASTE YOUR TOKEN HERE

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)

# Create public URL
tunnel = ngrok.connect(8000)
url = str(tunnel).replace("<NgrokTunnel:", "").replace(">", "").strip()
print(f"\n🎉 YOUR SERVER URL:")
print(url)
print("\n📋 COPY THE URL ABOVE AND PASTE IN THE APP!")

# Change to LivePortrait directory
import os
os.chdir("/content/LivePortrait")

# Write server code
server_code = '''
import os, sys, io, base64, uuid, json, logging
os.chdir("/content/LivePortrait")
sys.path.insert(0, "/content/LivePortrait")

logging.basicConfig(level=logging.INFO)

import torch
import cv2
import numpy as np
from PIL import Image
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware

from src.live_portrait_wrapper import LivePortraitWrapper
from src.config.inference_config import InferenceConfig
from src.config.crop_config import CropConfig
from src.utils.cropper import Cropper
from src.utils.io import resize_to_limit

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

print("Loading LivePortrait models...")
inf_cfg = InferenceConfig()
inf_cfg.flag_force_cpu = True
inf_cfg.flag_do_torch_compile = False
crop_cfg = CropConfig()
crop_cfg.flag_force_cpu = True

wrapper = LivePortraitWrapper(inf_cfg)
cropper = Cropper(crop_cfg=crop_cfg, flag_force_cpu=True)
print("✅ Models loaded!")

sessions = {}

@app.get("/health")
async def health():
    return {"status": "ok", "models_loaded": True}

@app.get("/")
async def root():
    return {"message": "Mickey Jagger Backend", "version": "1.0"}

@app.post("/upload/portrait")
async def upload_portrait(file: UploadFile = File(...)):
    contents = await file.read()
    img = Image.open(io.BytesIO(contents)).convert("RGB")
    img_rgb = np.array(img)
    img_rgb = resize_to_limit(img_rgb, inf_cfg.source_max_dim, inf_cfg.source_division)
    
    crop_result = cropper.crop_source_image(img_rgb, crop_cfg)
    if crop_result is None:
        return {"error": "No face detected"}
    
    cropped = crop_result["img_crop"]
    M_c2o = crop_result["M_c2o"]
    img_crop_256 = cv2.resize(cropped, (256, 256))
    x = img_crop_256.astype(np.float32) / 255.
    x = np.clip(x, 0, 1)
    x = torch.from_numpy(x).permute(2, 0, 1).unsqueeze(0)
    
    kp_info = wrapper.get_kp_info(x)
    
    session_id = str(uuid.uuid4())
    sessions[session_id] = {"tensor": x, "kp": kp_info, "img": img_rgb}
    
    return {"session_id": session_id, "success": True}

@app.post("/animate")
async def animate(session_id: str, pitch: float = 0, yaw: float = 0, roll: float = 0):
    if session_id not in sessions:
        return {"error": "Session not found"}
    
    s = sessions[session_id]
    
    with torch.no_grad():
        motion = {"pitch": pitch, "yaw": yaw, "roll": roll}
        result = wrapper.execute(s["tensor"], s["kp"], motion)
    
    result_img = (result[0].permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    result_img = cv2.resize(result_img, (s["img"].shape[1], s["img"].shape[0]))
    
    _, buffer = cv2.imencode(".jpg", cv2.cvtColor(result_img, cv2.COLOR_RGB2BGR))
    
    return {"frame": base64.b64encode(buffer).decode(), "success": True}

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open("/content/server.py", "w") as f:
    f.write(server_code)

# Start server
import subprocess
subprocess.Popen(["python", "/content/server.py"])
print("🚀 Server starting... wait 30 seconds!")

In [ ]:
# 📊 Check Server Status
import urllib.request, time
time.sleep(15)
try:
    r = urllib.request.urlopen("http://localhost:8000/health", timeout=5)
    print("✅ SERVER RUNNING!")
    print(r.read().decode())
except Exception as e:
    print(f"⏳ Still starting: {e}")
    print("Wait 30 more seconds and run again!")